In [20]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

In [21]:
df = pd.read_csv(r"D:\DEPI project\data\processed\df_final.csv")

In [22]:
X = df.drop(columns=["collision_severity"])
y = df["collision_severity"].map({"Slight": 0, "Serious": 1, "Fatal": 2})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [23]:
binary_ordinal_cols = [
    "urban_or_rural_area",
    "trunk_road_flag",
    "light_conditions",
    "road_surface_conditions",
]

ordinal_categories = [
    ["Rural", "Urban"],  
    ["Non-trunk", "Trunk (Roads managed by Highways England)"], 
    [
        "Daylight",
        "Darkness - lights lit",
        "Darkness - lighting unknown",
        "Darkness - Unlit or No Lighting",
    ], 
    ["Dry", "Wet or Flooded", "Snow or Ice"], 
]

nominal_cols = [
    "first_road_class",
    "second_road_class",
    "road_type",
    "junction_detail",
    "pedestrian_crossing",
    "weather_conditions",
    "did_police_officer_attend_scene_of_accident",
]

bool_cols = [
    "has_two_wheeler",
    "any_overturned",
    "any_left_road",
    "has_pedestrian",
    "is_weekend",
]

scaled_num_cols = [
    "longitude",
    "latitude",
    "number_of_vehicles",
    "speed_limit",
    "hour",
    "month",
    "max_engine_cc",
    "mean_casualty_age",
]

In [24]:
def transform_features(df):
  df = df.copy()
  df["number_of_vehicles"] = df["number_of_vehicles"].clip(upper=8)
  df["max_engine_cc"] = np.log1p(df["max_engine_cc"].clip(upper=20000))
  df["mean_casualty_age"] = df["mean_casualty_age"].clip(lower=0, upper=100)
  
  for col in bool_cols:
    df[col] = df[col].astype(int)
  return df

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num_scaler", StandardScaler(), scaled_num_cols),
        (
            "ord_bin",
            OrdinalEncoder(
                categories=ordinal_categories,
                dtype=int,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            binary_ordinal_cols,
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore", sparse_output=False, dtype=int
            ),
            nominal_cols,
        ),
        ("bool_pass", "passthrough", bool_cols),
    ]
)

In [26]:
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

In [27]:
full_pipeline = Pipeline(
    steps=[
        (
            "clip_and_log",
            FunctionTransformer(transform_features, validate=False),
        ),
        ("encode_and_scale", preprocessor),
        ("model", best_model),
    ]
)

In [28]:
full_pipeline.fit(X_train, y_train)

y_pred = full_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(full_pipeline, "accident_severity_pipeline.joblib")

              precision    recall  f1-score   support

           0       0.84      0.65      0.73     14935
           1       0.37      0.56      0.44      5032
           2       0.08      0.29      0.12       290

    accuracy                           0.62     20257
   macro avg       0.43      0.50      0.43     20257
weighted avg       0.71      0.62      0.65     20257



['accident_severity_pipeline.joblib']